In [1]:
import os
os.environ["JULIA_DEPOT_PATH"] = "/vol/bitbucket/jhl323/.julia"

from juliacall import Main as jl

# --- 1. Install Required Julia Packages ---
print("Initializing Julia environment and installing dependencies...")
jl.seval('import Pkg')

# Check if packages are installed, only install if missing
try:
    jl.seval('using JuMP, PATHSolver, LinearAlgebra')
    print("Julia dependencies loaded successfully.")
except:
    print("Installing missing Julia packages (this may take a few minutes)...")
    jl.Pkg.add("JuMP")
    jl.Pkg.add("PATHSolver")
    jl.Pkg.add("LinearAlgebra")
    jl.seval('using JuMP, PATHSolver, LinearAlgebra')
    print("Julia dependencies installed and loaded.")

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
Initializing Julia environment and installing dependencies...
Julia dependencies loaded successfully.


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from GridWorld import GridWorldEnv
from SRQagent import SRQAgent

In [3]:
def train_experiment(n_episodes=3000, p_env=0.8):
    env = GridWorldEnv(p=p_env)
    
    # Get environment dimensions
    num_agents = 2
    num_actions = len(env.action_space)
    
    # Initialize Agents with correct parameters
    agents = [
        SRQAgent(
            agent_id=0, 
            num_agents=num_agents,
            num_actions=num_actions,
            epsilon_robust=1.0,
            epsilon_explore=1.0,
            alpha=0.1,
            gamma=0.9,
            decay_rate=0.999  # Changed from 'decay' to 'decay_rate'
        ),
        SRQAgent(
            agent_id=1,
            num_agents=num_agents,
            num_actions=num_actions,
            epsilon_robust=1.0,
            epsilon_explore=1.0,
            alpha=0.1,
            gamma=0.9,
            decay_rate=0.999
        )
    ]
    
    history_rewards = [[], []]
    
    print(f"Starting Training: {n_episodes} Episodes, p={p_env}")
    
    for ep in tqdm(range(n_episodes)):
        obs = env.reset()
        done = False
        ep_rewards = [0, 0]
        
        while not done:
            actions = [ag.act(obs) for ag in agents]
            next_obs, rewards, done, _ = env.step(actions)
            
            for i, ag in enumerate(agents):
                ag.update(obs, actions, rewards, next_obs)
                
            obs = next_obs
            ep_rewards[0] += rewards[0]
            ep_rewards[1] += rewards[1]
            
        history_rewards[0].append(ep_rewards[0])
        history_rewards[1].append(ep_rewards[1])
        
        # Call decay_parameters() not decay_params()
        for ag in agents:
            ag.decay_parameters()
            
    return history_rewards

rewards_data = train_experiment(n_episodes=1000, p_env=0.8)

Loading Julia solver from bimatrix_asym_solver.jl...
Warming up JIT compilation (this may take a few seconds)...
Julia Solver Ready.
Loading Julia solver from bimatrix_asym_solver.jl...
Warming up JIT compilation (this may take a few seconds)...
Julia Solver Ready.
Starting Training: 1000 Episodes, p=0.8


  1%|          | 6/1000 [00:10<30:04,  1.82s/it]


ValueError: probabilities do not sum to 1

In [ ]:
def plot_results(rewards, window=50):
    fig, axes = plt.subplots(2, 1, figsize=(10, 12))
    
    for agent_id in [0, 1]:
        ax = axes[agent_id]
        data = rewards[agent_id]
        
        # Raw Data (scatter)
        ax.scatter(range(len(data)), data, s=2, alpha=0.3, color='gray', label='Episode Reward')
        
        # Rolling Average
        rolling_mean = np.convolve(data, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(data)), rolling_mean, linewidth=2, color='blue', label=f'Rolling Avg ({window})')
        
        # Statistics (Last 100 episodes)
        last_100 = data[-100:]
        mean_val = np.mean(last_100)
        std_val = np.std(last_100)
        
        ax.axhline(mean_val, color='red', linestyle='--', label=f'Final Mean: {mean_val:.2f}')
        
        ax.set_title(f"Agent {agent_id+1} Rewards (SRQ)")
        ax.set_xlabel("Episode")
        ax.set_ylabel("Total Reward")
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_results(rewards_data)